# Code Generator

Comments and docstrings

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important - Pause Endpoints when not in use</h1>
            <span style="color:#900;">
            If you do decide to use HuggingFace endpoints for this project, you should stop or pause the endpoints when you are done to avoid accruing unnecessary running cost. The costs are very low as long as you only run the endpoint when you're using it. Navigate to the HuggingFace endpoint UI <a href="https://ui.endpoints.huggingface.co/">here,</a> open your endpoint, and click Pause to put it on pause so you no longer pay for it.  
Many thanks to student John L. for raising this.
<br/><br/>
In week 8 we will use Modal instead of HuggingFace endpoints; with Modal you only pay for the time that you use it and you should get free credits.
            </span>
        </td>
    </tr>
</table>

In [1]:
# imports

import os
import io
import sys
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI
import google.generativeai
import anthropic
from IPython.display import Markdown, display, update_display
import gradio as gr
import subprocess
import google.generativeai as genai
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import transformers
import torch
from huggingface_hub import InferenceClient, login
#import ollama
#!ollama pull llama3.2:1b

In [2]:
# environment

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['ANTHROPIC_API_KEY'] = os.getenv('ANTHROPIC_API_KEY', 'your-key-if-not-using-env')
os.environ['DEEPSEEK_API_KEY'] = os.getenv('DEEPSEEK_API_KEY', 'your-key-if-not-using-env')
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY', 'your-key-if-not-using-env')
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN', 'your-key-if-not-using-env')
login(os.environ['HF_TOKEN'], add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
# initialize

openai = OpenAI()
claude = anthropic.Anthropic()
google.generativeai.configure()
OPENAI_MODEL = "gpt-4o"
CLAUDE_MODEL = "claude-3-5-sonnet-20240620"
GEMINI_MODEL = "gemini-2.0-flash"
LLAMA_MODEL = "codellama/CodeLlama-7b-hf"
LLAMA_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"

#LLAMA_MODEL = "llama3.2:1b"
CODE_LLAMA_URL = "https://s9u5lj7g824qidsh.eu-west-1.aws.endpoints.huggingface.cloud"

OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type": "application/json"}
OLLAMA_MODEL = "llama3.2"

In [4]:
system_message = "You are a code assistant designed to add docstrings and helpful comments to code for documentation purposes."
system_message += "Respond back with properly formatted code, including docstrings and comments. Keep comments concise. Do not add unnecessary code. Only focus on the code the user will provide. "
system_message += "Note that the code can be python, c++ or anything, so take into account formating. "
system_message += "Please, remove any header or foot like, such as ```python or ```cpp, etc. at the beginning of your response or ``` at the end of your response. It is mandatory. Please remove! "
system_message += "Do not respond with greetings, or any such extra output"

In [5]:
def user_prompt_for(code):
    user_prompt = "Rewrite this code to include helpful comments and docstrings. "
    user_prompt += "Respond only with code.\n"
    user_prompt += "Please, remove any header or foot like such as ```python or ```cpp at the beginning of your response or ``` at the end of your response. It is mandatory. Please remove! "
    user_prompt += "This is the code that you have to complete with comments and docstrings, try to be concise.\n\n"
    user_prompt += code
    return user_prompt

In [6]:
def messages_for(code):
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt_for(code)}
    ]

In [7]:
def comments_gpt(code):    
    stream = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages_for(code), stream=True)
    reply = ""
    #for chunk in stream:
    #    fragment = chunk.choices[0].delta.content or ""
    #    print(fragment, end='', flush=True)
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        yield reply #.replace('```cpp\n','').replace('```','')

In [8]:
def comments_claude(code):
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        system=system_message,
        messages=[{"role": "user", "content": user_prompt_for(code)}],
    )
    reply = ""
    #with result as stream:
    #    for text in stream.text_stream:
    #        print(text, end="", flush=True)
    with result as stream:
        for text in stream.text_stream:
            reply += text
            yield reply #.replace('```cpp\n','').replace('```','')

In [9]:
def comments_gemini(code):
    model = genai.GenerativeModel(GEMINI_MODEL)
    prompt = user_prompt_for(code)
    result = model.generate_content(prompt, stream=True)
    reply = ""
    #for chunk in result:
    #    text = chunk.text or ""        
    #    print(text, end='', flush=True)
    for chunk in result:
        reply += chunk.text or ""
        yield reply 

In [10]:
#def comments_llama(code):
#    tokenizer = AutoTokenizer.from_pretrained(LLAMA_MODEL)
#    #model = AutoModelForCausalLM.from_pretrained(LLAMA_MODEL)
#    pipe = pipeline("text-generation", model=LLAMA_MODEL, torch_dtype=torch.float16, device_maps="auto") #, tokenizer=tokenizer)
#    client = InferenceClient(model=LLAMA_MODEL) #, token=HF_TOKEN)  
#    messages = messages_for(code)
#    #result = client.text_generation(messages, stream=True, max_new_tokens=300, temperature=0.7)
#    #result = pipe(messages, max_new_tokens=2000, do_sample=True, temperature=0.7, stream=True)[0]["generated_text"]
#    result = pipe(messages, eos_token_id=tokenizer.eos_token_id, max_length=200, do_sample=True, temperature=0.7, stream=True, num_return_sequences=1)["generated_text"]
#    reply = ""
#    for chunk in result:
#        text = chunk.text or ""        
#        print(text, end='', flush=True)

In [11]:
#def comments_llama(code):
#    tokenizer = AutoTokenizer.from_pretrained(LLAMA_MODEL)
#    messages = messages_for(code)
#    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#    client = InferenceClient(CODE_LLAMA_URL, token=os.environ['HF_TOKEN'])
#    result = client.text_generation(text, stream=False, max_new_tokens=3000)
#    reply = ""
#    for chunk in result:
#        text = chunk.text or ""        
#        print(text, end='', flush=True)

#for r in stream:
#        result += r.token.text
#        yield result  

In [12]:
#def comments_ollama(code):
#    ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
#    result = ollama_via_openai.chat.completions.create(model=OLLAMA_MODEL, messages=messages_for(code), stream=True)
#    response = ""
#    #for chunk in stream:
#    #    response += chunk.choices[0].delta.content or ""
#    #    yield response
#    #return response
#    for chunk in result:
#        text = chunk.text or ""        
#        print(text, end='', flush=True)

In [13]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(100_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [14]:
python_hard = """# Be careful to support large number sizes

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [15]:
#comments_gpt(pi)

In [16]:
#comments_claude(pi)

In [17]:
#comments_gemini(pi)

In [18]:
#comments_ollama(pi)

In [19]:
def add_comments(code, model):
    if model=="GPT":
        result = comments_gpt(code)
    elif model=="Claude":
        result = comments_claude(code)
    elif model=="Gemini":
        result = comments_gemini(code)
    else:
        raise ValueError("Unknown model")
    
    for stream_so_far in result:
        yield stream_so_far        

In [20]:
css = """
.without {background-color: #306998; label: black}
.with {background-color: #050; label: black}
"""

In [21]:
#css = """
#/* Default Light Theme Styling */
#.without {
#    background-color: #f0f0f0;  /* Light gray */
#}
#.with {
#    background-color: #e3f2fd;  /* Soft light blue */
#}
#"""


with gr.Blocks(css=css) as ui:
    with gr.Row():
        #without_comments = gr.Textbox(label="Insert Code:", lines=10, value=python_hard)
        #with_comments = gr.Textbox(label="Code with comments:", lines=10)
        without_comments = gr.TextArea(label="Insert code:", elem_classes=["without"], lines=20)
        with_comments = gr.TextArea(label="Code with comments:", elem_classes=["with"], lines=20)
    with gr.Row():
        model = gr.Dropdown(["GPT", "Claude", "Gemini"], label="Select model", value="GPT")
        process = gr.Button("Add comments and docstrings")

    process.click(add_comments, inputs=[without_comments, model], outputs=[with_comments])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.
